# HomeReach Melbourne — 11: Diagnose the fixed Ridge forecast errors

**Research only. Reuse saved validation forecasts; fit zero new models.**

Notebook 10 ran successfully but its fixed Ridge model had higher validation MAE than last observation. This notebook asks **where that difference comes from**, not how to make the result disappear.

- Same 31 LGAs and 19 validation targets per LGA: **2019Q1–2023Q3**.
- The **2023Q4–2025Q3 final period stays unscored**. No rental-history values or transport tables are loaded.
- Saved outcomes are used only for retrospective error diagnosis. Direction of the future rent change is **not known at forecast time**.
- No parameter search, model fitting, new forecasts, row removal, statistical-significance claim or application change.
- The diagnostic plan was specified **after notebook 10's overall result was known**. It is further use of the development/validation set, not a fresh independent test.

Keep `feat/rental-forecast-baselines` active. Select **homereach** and run the nine code cells in order. Notebook 10's CSVs and manifest must already exist. Do not rerun notebooks 09/10 just to use this notebook. The final period has appeared in prior descriptive EDA, so it is reserved from modelling assessment, not a pristine blind test.

## 1. Verify saved evidence before loading it

Check the two earlier protocols and their manifests, the source fingerprint and the saved CSV fingerprints. File hashing checks integrity but does not parse the original rental history or score final-period values.

The analysis loads validation predictions, historical-input lineage and earlier validation metrics. A mismatch stops execution. Do not edit a fingerprint or delete evidence to get past a failed check. No Internet access or package installation occurs.

In [1]:
print("Starting notebook 11 — saved-error diagnostics...", flush=True)
from pathlib import Path
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def file_hash(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()


def read_json(path):
    result = json.loads(Path(path).read_text(encoding='utf-8-sig'))
    require(isinstance(result, dict), f'Expected JSON object: {path}')
    return result


def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'pyproject.toml').is_file() and (p / 'data').is_dir()), None)
require(ROOT is not None, 'Open this notebook inside the existing HomeReach project.')
BRANCH = subprocess.run(['git', '-C', str(ROOT), 'branch', '--show-current'],
                        capture_output=True, text=True, check=True, timeout=10).stdout.strip()
require(BRANCH == 'feat/rental-forecast-baselines',
        'Use the research branch feat/rental-forecast-baselines; do not run on deployed main.')
BASE_DIR = ROOT / 'data/interim/rental_forecast_baselines'
RIDGE_DIR = ROOT / 'data/interim/rental_ridge_validation'
OUT = ROOT / 'data/interim/rental_ridge_diagnostics'
REPORT_PATH = ROOT / 'docs/rental_ridge_diagnostics.md'
PROTOCOL_PATH = ROOT / 'config/rental_ridge_diagnostic_protocol.json'
BASE_CONFIG = ROOT / 'config/rental_forecast_protocol.json'
RIDGE_CONFIG = ROOT / 'config/rental_ridge_protocol.json'
BASE_MANIFEST = BASE_DIR / 'baseline_manifest.json'
RIDGE_MANIFEST = RIDGE_DIR / 'ridge_manifest.json'
for path in [PROTOCOL_PATH, BASE_CONFIG, RIDGE_CONFIG, BASE_MANIFEST, RIDGE_MANIFEST]:
    require(path.is_file(), f'Missing {path.relative_to(ROOT)}; use the completed local notebooks 09/10.')
plan = read_json(PROTOCOL_PATH)
require(canonical_hash(plan) == '98af177f32c9718639f2a1a2dbb293d9e28bca952b617218fb7495be56fb5168', 'The diagnostic plan changed; preserve this experiment before reviewing a new plan.')
base_protocol, ridge_protocol = read_json(BASE_CONFIG), read_json(RIDGE_CONFIG)
bm, rm = read_json(BASE_MANIFEST), read_json(RIDGE_MANIFEST)
SOURCE_HASH = 'b8e96347494f09fea2a2ace55997fd34c9398f8509cb3450d65444aebfaaa837'
require(bm['source_workbook_sha256'] == rm['source_workbook_sha256'] == SOURCE_HASH, 'Source vintages differ.')
require(bm['input_csv_sha256'] == rm['input_csv_sha256'], 'Baseline and Ridge rental inputs differ.')
require(canonical_hash(base_protocol) == bm['protocol_sha256'] == rm['baseline_protocol_sha256'], 'Baseline protocol mismatch.')
require(canonical_hash(ridge_protocol) == rm['ridge_protocol_sha256'], 'Ridge protocol mismatch.')
require(rm['baseline_manifest_sha256'] == file_hash(BASE_MANIFEST), 'Baseline evidence changed since the Ridge run.')
require(ridge_protocol['model_name'] == 'ridge_lags_seasonal_alpha1'
        and ridge_protocol['estimator'] == {'class':'Ridge','alpha':1.0,'fit_intercept':True,'solver':'svd'},
        'This diagnosis is for the saved fixed Ridge experiment only.')
for name in ['validation_start', 'validation_end', 'holdout_start', 'holdout_end', 'property_type']:
    require(base_protocol[name] == ridge_protocol[name] == plan[name], f'Scope mismatch: {name}')
require(bm['quality']['holdout_forecasts'] == bm['quality']['holdout_scores'] == 0, 'Baseline final period is not reserved.')
require(rm['quality']['final_period_forecasts'] == rm['quality']['final_period_scores'] == 0, 'Ridge final period is not reserved.')
require(rm['quality']['ridge_forecasts'] == 589 and rm['quality']['combined_forecasts'] == 1767
        and rm['quality']['csv_readback_matches'] is True
        and rm['quality']['protected_files_unchanged'] is True, 'The Ridge checkpoint is incomplete.')

# Snapshot protected files. Hashing bytes is not loading outcomes into the analysis.
protected = {PROTOCOL_PATH, BASE_CONFIG, RIDGE_CONFIG, BASE_MANIFEST, RIDGE_MANIFEST}
for directory in [BASE_DIR, RIDGE_DIR, ROOT/'app', ROOT/'deploy', ROOT/'src', ROOT/'config', ROOT/'scripts']:
    protected.update(p for p in directory.rglob('*') if p.is_file()
                     and '__pycache__' not in p.parts and p.suffix != '.pyc')
protected.update(p for p in (ROOT/'notebooks').glob('*.ipynb') if not p.name.startswith('11_'))
for relative in ['README.md', 'docs/data_registry.csv', 'docs/rental_forecast_baselines.md',
                 'docs/rental_ridge_validation.md', 'data/processed/rent_quarterly.csv',
                 'data/processed/rent_quality_report.json']:
    if (ROOT/relative).is_file(): protected.add(ROOT/relative)
input_hashes = {p: file_hash(p) for p in sorted(protected)}
rental_csv = ROOT/'data/processed/rent_quarterly.csv'
if rental_csv.is_file():
    require(input_hashes[rental_csv] == rm['input_csv_sha256'], 'Local rental CSV differs from the saved experiment.')


def read_verified_csv(directory, manifest, name):
    require(Path(name).name == name, 'Expected a plain CSV filename.')
    path = directory/name
    require(path.is_file(), f'Missing saved evidence: {path}')
    require(file_hash(path) == manifest['output_sha256'][name], f'Changed evidence: {name}. Do not bypass its manifest.')
    frame = pd.read_csv(path, dtype=manifest['output_schemas'][name], keep_default_na=False)
    require(frame.columns.tolist() == list(manifest['output_schemas'][name]), f'Schema changed: {name}')
    return frame


RIDGE_FILES = ['validation_predictions_all_methods.csv', 'validation_metrics_overall.csv',
               'validation_metrics_by_lga.csv', 'validation_metrics_by_quarter.csv', 'forecast_feature_lineage.csv']
tables = {name: read_verified_csv(RIDGE_DIR, rm, name) for name in RIDGE_FILES}
baseline_predictions = read_verified_csv(BASE_DIR, bm, 'validation_predictions.csv')
saved_predictions = tables['validation_predictions_all_methods.csv']
lineage = tables['forecast_feature_lineage.csv']
PERIODS = pd.period_range(plan['validation_start'], plan['validation_end'], freq='Q-DEC')
ALLOWED_QUARTERS = PERIODS.astype(str).tolist()
RIDGE = 'ridge_lags_seasonal_alpha1'
LAST = 'last_observation'
SEASONAL = 'seasonal_naive'
MODELS = [LAST, RIDGE, SEASONAL]
KEY = ['rental_area_name', 'property_type', 'target_quarter']
TOL = float(plan['numerical_tolerance_aud_week'])
if (OUT/'diagnostic_manifest.json').exists():
    old = read_json(OUT/'diagnostic_manifest.json')
    require(old['protocol_sha256'] == canonical_hash(plan)
            and old['ridge_manifest_sha256'] == input_hashes[RIDGE_MANIFEST],
            'Existing diagnostics use different evidence or settings; preserve them for review.')
print('DIAGNOSTIC INPUT CHECKPOINT PASSED')
print('Research branch:', BRANCH)
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)
print('Saved forecast rows:', len(saved_predictions), '| Baseline rows:', len(baseline_predictions))
print('Rental-history values loaded: 0 | New model fits: 0 | Final-period forecasts scored: 0')

Starting notebook 11 — saved-error diagnostics...
DIAGNOSTIC INPUT CHECKPOINT PASSED
Research branch: feat/rental-forecast-baselines
Python: 3.12.14 | pandas: 3.0.3
Saved forecast rows: 1767 | Baseline rows: 1178
Rental-history values loaded: 0 | New model fits: 0 | Final-period forecasts scored: 0


## 2. Define paired errors and their additive contributions

One comparison row is the **same LGA, property category and target quarter**, with the same actual outcome and source cell for all models. We join by those keys, never by row position.

For each target, define:

$$d_i = |\hat y_{i,\mathrm{Ridge}}-y_i|-|\hat y_{i,\mathrm{last}}-y_i|.$$

Positive $d_i$ means Ridge made the larger absolute error; negative means Ridge helped. The average of all 589 differences is exactly the overall MAE gap. A group's contribution is $\sum_{i\in group}d_i/589$, not an unweighted average of group averages. Negative contributions offset positive ones.

The sign of `actual - last_observation_forecast` describes whether the **published median** rose, fell or was unchanged. It does not describe every property. Its use here is retrospective: an operational forecast cannot choose its model using that unknown future outcome. These are out-of-sample validation forecast errors, not fitted training residuals.

In [2]:
REQUIRED = ['model', *KEY, 'forecast_origin_quarter', 'forecast_weekly_rent_aud',
            'actual_weekly_rent_aud', 'target_source_median_cell',
            'error_aud_week', 'absolute_error_aud_week', 'squared_error']


def validate_predictions(frame, allowed_quarters):
    require(set(REQUIRED).issubset(frame.columns), 'Missing forecast evidence columns.')
    require(len(frame) > 0, 'No prediction rows.')
    require(not frame.duplicated(['model', *KEY]).any(), 'Duplicate model/target key.')
    for col in ['model', *KEY, 'forecast_origin_quarter', 'target_source_median_cell']:
        require(frame[col].notna().all() and frame[col].astype('string').str.len().gt(0).all(), f'Blank identifier: {col}')
    require(set(frame['model']) == set(MODELS), 'Missing or unexpected model.')
    require(frame['property_type'].eq('2 bedroom flats').all(), 'Unexpected property category.')
    require(frame['target_quarter'].isin(allowed_quarters).all(), 'Target outside validation; final period is excluded.')
    target = pd.PeriodIndex(frame['target_quarter'], freq='Q-DEC')
    origin = pd.PeriodIndex(frame['forecast_origin_quarter'], freq='Q-DEC')
    require(np.array_equal(origin.asi8, target.asi8 - 1), 'Origin is not the preceding quarter.')
    for col in ['forecast_weekly_rent_aud', 'actual_weekly_rent_aud',
                'error_aud_week', 'absolute_error_aud_week', 'squared_error']:
        require(np.isfinite(frame[col].to_numpy(dtype=float)).all(), f'Non-finite numeric field: {col}')
    errors = frame['forecast_weekly_rent_aud'].to_numpy(float) - frame['actual_weekly_rent_aud'].to_numpy(float)
    np.testing.assert_allclose(errors, frame['error_aud_week'], rtol=1e-10, atol=1e-9)
    np.testing.assert_allclose(np.abs(errors), frame['absolute_error_aud_week'], rtol=1e-10, atol=1e-9)
    np.testing.assert_allclose(errors**2, frame['squared_error'], rtol=1e-10, atol=1e-8)
    reference = frame.loc[frame.model.eq(LAST)].sort_values(KEY).reset_index(drop=True)
    for model in MODELS:
        part = frame.loc[frame.model.eq(model)].sort_values(KEY).reset_index(drop=True)
        require(part[KEY].equals(reference[KEY]), f'Unequal target coverage: {model}')
        require(part['forecast_origin_quarter'].equals(reference['forecast_origin_quarter']), 'Unequal origins.')
        require(part['target_source_median_cell'].equals(reference['target_source_median_cell']), 'Unequal target source cells.')
        np.testing.assert_allclose(part.actual_weekly_rent_aud, reference.actual_weekly_rent_aud, rtol=0, atol=0)
    return reference


def pair_forecasts(frame, allowed_quarters):
    ref = validate_predictions(frame, allowed_quarters)
    pairs = ref[KEY + ['forecast_origin_quarter', 'actual_weekly_rent_aud', 'target_source_median_cell']].copy()
    for model, prefix in [(LAST, 'last'), (RIDGE, 'ridge'), (SEASONAL, 'seasonal')]:
        part = frame.loc[frame.model.eq(model), KEY + ['forecast_weekly_rent_aud']]
        pairs = pairs.merge(part.rename(columns={'forecast_weekly_rent_aud':prefix+'_forecast'}),
                            on=KEY, how='left', validate='one_to_one', sort=False)
        pairs[prefix+'_error'] = pairs[prefix+'_forecast'] - pairs['actual_weekly_rent_aud']
        pairs[prefix+'_absolute_error'] = pairs[prefix+'_error'].abs()
    pairs['actual_change'] = pairs['actual_weekly_rent_aud'] - pairs['last_forecast']
    pairs['ridge_implied_change'] = pairs['ridge_forecast'] - pairs['last_forecast']
    pairs['ridge_minus_last_absolute_error'] = pairs['ridge_absolute_error'] - pairs['last_absolute_error']
    delta = pairs['ridge_minus_last_absolute_error']
    pairs['paired_result'] = np.select([delta.lt(-TOL), delta.gt(TOL)], ['Ridge lower error','Last observation lower error'], default='Numerical tie')
    change = pairs['actual_change']
    pairs['observed_change_direction'] = np.select([change.lt(-TOL), change.gt(TOL)], ['decreased','increased'], default='unchanged')
    pairs['target_year'] = pd.PeriodIndex(pairs.target_quarter, freq='Q-DEC').year.astype('int64')
    return pairs.sort_values(KEY).reset_index(drop=True)


def summarise_pairs(frame, grouping):
    require(len(frame) > 0, 'No comparison rows to summarise.')
    cols = [grouping] if isinstance(grouping, str) else grouping
    grouper = cols[0] if len(cols) == 1 else cols
    rows = []
    for labels, part in frame.groupby(grouper, sort=True, observed=True, dropna=False):
        labels = (labels,) if len(cols) == 1 else labels
        diff = part['ridge_minus_last_absolute_error']
        rows.append({**dict(zip(cols, labels)), 'n_targets':len(part),
            'last_mae':float(part.last_absolute_error.mean()),
            'ridge_mae':float(part.ridge_absolute_error.mean()),
            'ridge_minus_last_mae':float(diff.mean()),
            'sum_paired_loss_difference':float(diff.sum()),
            'contribution_to_overall_mae_gap':float(diff.sum()/len(frame)),
            'ridge_lower_error_targets':int(diff.lt(-TOL).sum()),
            'last_lower_error_targets':int(diff.gt(TOL).sum()),
            'numerical_ties':int(diff.abs().le(TOL).sum())})
    result = pd.DataFrame(rows)
    require(int(result.n_targets.sum()) == len(frame), 'Group counts do not cover every target.')
    np.testing.assert_allclose(result.contribution_to_overall_mae_gap.sum(),
                               frame.ridge_minus_last_absolute_error.mean(), rtol=1e-10, atol=1e-10)
    return result


def metrics_from_predictions(frame, group_columns):
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for label, part in frame.groupby(grouper, sort=True, observed=True):
        labels = (label,) if len(group_columns) == 1 else label
        e = part.forecast_weekly_rent_aud.to_numpy(float) - part.actual_weekly_rent_aud.to_numpy(float)
        rows.append({**dict(zip(group_columns, labels)), 'n_forecasts':len(e),
                     'mae_aud_week':float(np.mean(np.abs(e))), 'rmse_aud_week':float(np.sqrt(np.mean(e**2))),
                     'mean_signed_error_aud_week':float(e.mean())})
    return pd.DataFrame(rows)


def assert_same_table(left, right, key):
    require(left.columns.tolist() == right.columns.tolist(), 'Table columns changed.')
    pd.testing.assert_frame_equal(left.sort_values(key).reset_index(drop=True),
                                  right.sort_values(key).reset_index(drop=True),
                                  check_dtype=False, check_exact=False, rtol=1e-10, atol=1e-9)

## 3. Test the diagnostic code with fictional forecasts

These checks do not train a model or alter your saved predictions. They test key matching, outcome consistency, incorrect dates, invalid numbers, preserved rows and exact additive accounting. A passing test suite does not mean Ridge is a better model.

In [3]:
self_checks = []

def check(name, fn):
    fn(); self_checks.append(name)


def rejected(action):
    try:
        action()
    except (ValueError, AssertionError):
        return
    raise AssertionError('An intentionally invalid input was accepted.')


def example_frame():
    rows=[]
    periods=pd.period_range('2019Q1','2019Q4',freq='Q-DEC')
    actual=[120.,90.,100.,110.]
    forecasts={LAST:[100.]*4,RIDGE:[105.,95.,101.,108.],SEASONAL:[98.]*4}
    for model in MODELS:
        for i,q in enumerate(periods):
            f=forecasts[model][i]; e=f-actual[i]
            rows.append({'model':model,'rental_area_name':'Fictional A','property_type':'2 bedroom flats',
                'target_quarter':str(q),'forecast_origin_quarter':str(q-1),
                'forecast_weekly_rent_aud':f,'actual_weekly_rent_aud':actual[i],
                'target_source_median_cell':f'A{i+1}','error_aud_week':e,
                'absolute_error_aud_week':abs(e),'squared_error':e**2})
    return pd.DataFrame(rows)

sample=example_frame(); qs=['2019Q1','2019Q2','2019Q3','2019Q4']
sp=pair_forecasts(sample,qs)
check('known paired errors',lambda:np.testing.assert_allclose(sp.ridge_minus_last_absolute_error,[-5.,-5.,1.,-8.]))
check('duplicate target rejected',lambda:rejected(lambda:pair_forecasts(pd.concat([sample,sample.iloc[[0]]]),qs)))
check('missing model target rejected',lambda:rejected(lambda:pair_forecasts(sample.iloc[1:].copy(),qs)))
bad=sample.copy();bad.loc[4,'actual_weekly_rent_aud']=121.;bad.loc[4,'error_aud_week']=-16.;bad.loc[4,'absolute_error_aud_week']=16.;bad.loc[4,'squared_error']=256.
check('different actuals rejected',lambda:rejected(lambda:pair_forecasts(bad,qs)))
bad=sample.copy();bad.loc[4,'target_source_median_cell']='Z99'
check('different source cells rejected',lambda:rejected(lambda:pair_forecasts(bad,qs)))
bad=sample.copy();bad.loc[0,'forecast_origin_quarter']='2019Q1'
check('invalid forecast origin rejected',lambda:rejected(lambda:pair_forecasts(bad,qs)))
bad=sample.copy();bad.loc[0,'target_quarter']='2023Q4'
check('final-period target rejected',lambda:rejected(lambda:pair_forecasts(bad,qs)))
bad=sample.copy();bad.loc[0,'forecast_weekly_rent_aud']=np.inf
check('non-finite forecast rejected',lambda:rejected(lambda:pair_forecasts(bad,qs)))
check('input order does not affect matching',lambda:pd.testing.assert_frame_equal(sp,pair_forecasts(sample.sample(frac=1,random_state=19),qs)))
check('group contributions reconcile',lambda:summarise_pairs(sp,'observed_change_direction'))
check('unchanged median gives exact last-observation prediction',lambda:require(sp.loc[sp.observed_change_direction.eq('unchanged'),'last_absolute_error'].eq(0).all(),'Unchanged example failed.'))
copy=sample.copy(deep=True);pair_forecasts(sample,qs)
check('input predictions not modified',lambda:pd.testing.assert_frame_equal(sample,copy))
print('Diagnostic self-check groups:',len(self_checks),'passed')

Diagnostic self-check groups: 12 passed


## 4. Build the 589 paired targets and reconcile previous metrics

This is verification of saved predictions, **not a new forecasting run**. All three methods must have exactly the same 31 × 19 targets. The earlier baseline predictions must remain identical. The last-observation forecast must equal the verified lag-1 input recorded by notebook 10.

The original overall, per-LGA and per-quarter metric tables are recomputed from the saved rows, not copied as trusted answers. The conditional analyses below cannot change the overall 589-target result.

In [4]:
validate_predictions(saved_predictions, ALLOWED_QUARTERS)
lgas=sorted(saved_predictions.rental_area_name.unique())
require(len(lgas)==31 and len(saved_predictions)==1767,'Expected 31 LGAs and 1,767 saved prediction rows.')
expected_keys={(lga,'2 bedroom flats',q) for lga in lgas for q in ALLOWED_QUARTERS}
for model in MODELS:
    observed=set(saved_predictions.loc[saved_predictions.model.eq(model),KEY].itertuples(index=False,name=None))
    require(observed==expected_keys,f'Incomplete validation grid for {model}.')
base_columns=REQUIRED
old=baseline_predictions[base_columns].copy()
current=saved_predictions.loc[saved_predictions.model.isin([LAST,SEASONAL]),base_columns]
assert_same_table(old,current,['model',*KEY])
pairs=pair_forecasts(saved_predictions,ALLOWED_QUARTERS)

require(len(lineage)==2356 and not lineage.duplicated([*KEY,'feature']).any(),'Feature lineage changed.')
require(lineage.target_quarter.isin(ALLOWED_QUARTERS).all(),'Lineage contains out-of-scope targets.')
lag1=lineage.loc[lineage.feature.eq('rent_lag_1'),KEY+[
    'source_quarter','source_value_aud_week','source_file','source_sheet','source_median_cell']].copy()
require(len(lag1)==589,'Expected 589 lag-1 records.')
lag1=lag1.rename(columns={'source_quarter':'lag1_source_quarter','source_value_aud_week':'lag1_source_value',
                         'source_median_cell':'lag1_source_median_cell'})
pairs=pairs.merge(lag1,on=KEY,how='left',validate='one_to_one',indicator=True)
require(pairs['_merge'].eq('both').all(),'Missing lag-1 evidence.')
pairs=pairs.drop(columns='_merge')
require(pairs.lag1_source_quarter.eq(pairs.forecast_origin_quarter).all(),'Lag-1 source quarter differs from origin.')
np.testing.assert_allclose(pairs.last_forecast,pairs.lag1_source_value,rtol=0,atol=0)
prior_last=baseline_predictions.loc[baseline_predictions.model.eq(LAST)].sort_values(KEY).reset_index(drop=True)
current_pairs=pairs.sort_values(KEY).reset_index(drop=True)
for new_col,old_col in [('lag1_source_quarter','forecast_source_quarter'),
                        ('lag1_source_median_cell','forecast_source_median_cell'),
                        ('source_file','source_file'),('source_sheet','source_sheet')]:
    require(current_pairs[new_col].astype('string').equals(prior_last[old_col].astype('string')),
            f'Lag-1 source evidence differs from notebook 09: {new_col}')

np.testing.assert_allclose(pairs.ridge_error,pairs.ridge_implied_change-pairs.actual_change,rtol=0,atol=1e-10)
np.testing.assert_allclose(pairs.last_error,-pairs.actual_change,rtol=0,atol=1e-10)

metric_sets=[(['model'],'validation_metrics_overall.csv'),
             (['model','rental_area_name','property_type'],'validation_metrics_by_lga.csv'),
             (['model','target_quarter'],'validation_metrics_by_quarter.csv')]
for keys,name in metric_sets:
    fresh=metrics_from_predictions(saved_predictions,keys)
    assert_same_table(fresh[tables[name].columns],tables[name],keys)
overall_metrics=metrics_from_predictions(saved_predictions,['model'])
score=overall_metrics.set_index('model')
pooled_gap=float(score.loc[RIDGE,'mae_aud_week']-score.loc[LAST,'mae_aud_week'])
np.testing.assert_allclose(pairs.ridge_minus_last_absolute_error.mean(),pooled_gap,rtol=1e-10,atol=1e-10)
require(len(pairs)==589 and pairs.target_quarter.isin(ALLOWED_QUARTERS).all(),'Invalid paired scope.')
print('PAIRED EVIDENCE CHECKPOINT PASSED')
print('Paired validation targets:',len(pairs),'| Verified prior baseline forecasts:',len(current))
print('Verified lag-1 inputs:',len(lag1),'| Prior overall/LGA/quarter metrics: matched')
print('Ridge minus last-observation MAE:',round(pooled_gap,6),'AUD/week')
display(overall_metrics.round(4))

PAIRED EVIDENCE CHECKPOINT PASSED
Paired validation targets: 589 | Verified prior baseline forecasts: 1178
Verified lag-1 inputs: 589 | Prior overall/LGA/quarter metrics: matched
Ridge minus last-observation MAE: 2.240677 AUD/week


,model,n_forecasts,mae_aud_week,rmse_aud_week,mean_signed_error_aud_week
0,last_observation,589,10.9728,16.2303,-4.3582
1,ridge_lags_seasonal_alpha1,589,13.2135,18.6421,-1.9758
2,seasonal_naive,589,25.2937,38.9464,-13.0934


## 5. Locate the extra error without dropping any observations

Group by quarter, year and LGA, then separately by observed change direction. The year 2023 contains **Q1–Q3 only** (93 targets), whereas 2019–2022 each contain four quarters (124 targets). Do not compare unequal-year error sums as if their sample sizes were the same.

`contribution_to_overall_mae_gap` always uses **589** as its denominator. The contributions from a complete partition add to the overall MAE gap, even when some are negative. They are arithmetic descriptions, not causal effects.

The direction-based table uses the actual target outcome. A rule saying “use last observation when rent will be unchanged” is not deployable: the quoted condition is unknown when making the forecast. No outcome-dependent hybrid forecast is created here.

In [5]:
by_lga=summarise_pairs(pairs,['rental_area_name','property_type'])
by_quarter=summarise_pairs(pairs,'target_quarter')
by_year=summarise_pairs(pairs,'target_year')
by_change=summarise_pairs(pairs,'observed_change_direction')
require(len(by_lga)==31 and len(by_quarter)==19 and len(by_year)==5 and len(by_change)==3,'Unexpected group coverage.')
require(by_lga.n_targets.eq(19).all() and by_quarter.n_targets.eq(31).all(),'Unequal area/quarter coverage.')
for frame in [by_lga,by_quarter,by_year,by_change]:
    np.testing.assert_allclose(frame.contribution_to_overall_mae_gap.sum(),pooled_gap,rtol=1e-10,atol=1e-10)
    require((frame.ridge_lower_error_targets+frame.last_lower_error_targets+frame.numerical_ties).eq(frame.n_targets).all(),
            'Win/loss/tie counts do not reconcile.')
unchanged=pairs.loc[pairs.observed_change_direction.eq('unchanged')]
require(unchanged.last_absolute_error.le(TOL).all(),'Unchanged medians must reproduce the last observation.')
wins=int(pairs.ridge_minus_last_absolute_error.lt(-TOL).sum())
losses=int(pairs.ridge_minus_last_absolute_error.gt(TOL).sum())
ties=int(pairs.ridge_minus_last_absolute_error.abs().le(TOL).sum())
print('By observed change — retrospective diagnosis only:')
display(by_change.round(4))
print('By year — 2023 includes Q1-Q3 only:')
display(by_year.round(4))
print('Paired targets: Ridge lower error =',wins,'| Last observation lower error =',losses,'| Ties =',ties)
print('LGAs with lower Ridge MAE:',int(by_lga.ridge_minus_last_mae.lt(-TOL).sum()))
print('LGAs with lower last-observation MAE:',int(by_lga.ridge_minus_last_mae.gt(TOL).sum()))

By observed change — retrospective diagnosis only:


,observed_change_direction,n_targets,last_mae,ridge_mae,ridge_minus_last_mae,sum_paired_loss_difference,contribution_to_overall_mae_gap,ridge_lower_error_targets,last_lower_error_targets,numerical_ties
0,decreased,145,13.4345,17.7738,4.3394,629.2078,1.0683,48,97,0
1,increased,296,15.2534,14.8220,-0.4314,-127.6890,-0.2168,165,131,0
2,unchanged,148,0.0000,5.5287,5.5287,818.2402,1.3892,0,148,0


By year — 2023 includes Q1-Q3 only:


,target_year,n_targets,last_mae,ridge_mae,ridge_minus_last_mae,sum_paired_loss_difference,contribution_to_overall_mae_gap,ridge_lower_error_targets,last_lower_error_targets,numerical_ties
0,2019,124,8.0887,6.8671,-1.2216,-151.4833,-0.2572,61,63,0
1,2020,124,11.4919,15.3583,3.8664,479.4328,0.8140,47,77,0
2,2021,124,6.6210,9.5510,2.9300,363.3193,0.6168,36,88,0
3,2022,124,11.9355,13.0909,1.1554,143.2727,0.2432,52,72,0
4,2023,93,18.6452,23.8626,5.2174,485.2174,0.8238,17,76,0


Paired targets: Ridge lower error = 213 | Last observation lower error = 376 | Ties = 0
LGAs with lower Ridge MAE: 5
LGAs with lower last-observation MAE: 26


## 6. Inspect the largest disagreements with source references

These are explicitly **retrospective examples**: the ten largest error penalties and ten largest error savings for Ridge. They are not a separate evaluation set, representative sample, or licence to delete outliers. Every one of the 589 targets remains in all full-sample metrics.

The model’s *implied change* is `Ridge forecast − last observed rent`. It is a re-expression of the existing level forecast, not a newly trained change model. Comparing it with the observed change helps describe overshoots and corrections in the wrong direction; it does not prove their underlying cause.

In [6]:
case_columns=[*KEY,'forecast_origin_quarter','actual_weekly_rent_aud','last_forecast','ridge_forecast',
              'actual_change','ridge_implied_change','last_absolute_error','ridge_absolute_error',
              'ridge_minus_last_absolute_error','source_file','source_sheet',
              'lag1_source_quarter','lag1_source_median_cell','target_source_median_cell']
penalties=pairs.sort_values(['ridge_minus_last_absolute_error',*KEY],ascending=[False,True,True,True]).head(10)
savings=pairs.sort_values(['ridge_minus_last_absolute_error',*KEY],ascending=[True,True,True,True]).head(10)
case_examples=pd.concat([
    penalties[case_columns].assign(selection_reason='ten largest Ridge penalties; selected after observing errors'),
    savings[case_columns].assign(selection_reason='ten largest Ridge savings; selected after observing errors'),
],ignore_index=True)
require(len(case_examples)==20,'Unexpected case-example size.')
display(case_examples[[*KEY,'actual_change','ridge_implied_change','ridge_minus_last_absolute_error','selection_reason']].round(3))
summary=pd.DataFrame([{'paired_targets':len(pairs),'ridge_lower_error_targets':wins,
    'last_lower_error_targets':losses,'numerical_ties':ties,
    'ridge_lower_mae_lgas':int(by_lga.ridge_minus_last_mae.lt(-TOL).sum()),
    'last_lower_mae_lgas':int(by_lga.ridge_minus_last_mae.gt(TOL).sum()),
    'overall_mae_gap_aud_week':pooled_gap,'new_model_fits':0,'holdout_forecasts_scored':0}])

,rental_area_name,property_type,target_quarter,actual_change,ridge_implied_change,ridge_minus_last_absolute_error,selection_reason
0,Melbourne,2 bedroom flats,2020Q3,-50.0,59.426,59.426,ten largest Ridge penalties; selected after ob...
1,Port Phillip,2 bedroom flats,2020Q4,-10.0,35.503,35.503,ten largest Ridge penalties; selected after ob...
2,Melton,2 bedroom flats,2020Q1,0.0,28.325,28.325,ten largest Ridge penalties; selected after ob...
3,Stonnington,2 bedroom flats,2020Q3,-25.0,26.718,26.718,ten largest Ridge penalties; selected after ob...
4,Yarra,2 bedroom flats,2020Q3,-10.0,26.144,26.144,ten largest Ridge penalties; selected after ob...
5,Bayside,2 bedroom flats,2023Q2,10.0,-23.994,23.994,ten largest Ridge penalties; selected after ob...
6,Port Phillip,2 bedroom flats,2021Q1,0.0,23.637,23.637,ten largest Ridge penalties; selected after ob...
7,Stonnington,2 bedroom flats,2020Q4,-25.0,22.841,22.841,ten largest Ridge penalties; selected after ob...
8,Whitehorse,2 bedroom flats,2020Q3,-10.0,22.369,22.369,ten largest Ridge penalties; selected after ob...
9,Monash,2 bedroom flats,2020Q3,-5.0,22.348,22.348,ten largest Ridge penalties; selected after ob...


## 7. Four diagnostic charts

1. **MAE gap by quarter:** above zero favours last observation; below zero favours Ridge. Every quarter is shown.
2. **MAE gap by LGA:** every LGA has the same 19 targets. Sorting describes an error difference, not rental desirability.
3. **Error by observed change:** show sample sizes and compare means. Groups are only known after the outcome.
4. **Actual versus implied change:** each point is one target. The horizontal zero line is last observation’s no-change prediction; the diagonal is a perfect change prediction. Neither line is a new fitted model.

These are validation forecast errors, not independent random observations. No iid significance test, causal explanation or confidence interval is inferred from the plots. Plotly HTML exports retain hover/zoom interactions: [official documentation](https://plotly.com/python/interactive-html-export/).

In [7]:
figures={}
figures['01_mae_gap_by_quarter.html']=px.bar(
    by_quarter,x='target_quarter',y='ridge_minus_last_mae',
    title='Ridge minus last-observation MAE by target quarter',
    labels={'target_quarter':'Validation target quarter','ridge_minus_last_mae':'MAE difference (AUD/week)'},
    hover_data={'n_targets':True,'last_mae':':.3f','ridge_mae':':.3f','contribution_to_overall_mae_gap':':.4f'})
figures['01_mae_gap_by_quarter.html'].add_hline(y=0,line_dash='dash')
figures['01_mae_gap_by_quarter.html'].update_xaxes(tickangle=-45)
ordered=by_lga.sort_values('ridge_minus_last_mae')
figures['02_mae_gap_by_lga.html']=px.bar(
    ordered,x='ridge_minus_last_mae',y='rental_area_name',orientation='h',height=1000,
    title='Ridge minus last-observation MAE: all 31 LGAs',
    labels={'ridge_minus_last_mae':'MAE difference (AUD/week)','rental_area_name':'Rental-source LGA'},
    hover_data={'n_targets':True,'last_mae':':.3f','ridge_mae':':.3f'})
figures['02_mae_gap_by_lga.html'].add_vline(x=0,line_dash='dash')
figures['02_mae_gap_by_lga.html'].update_yaxes(categoryorder='array',categoryarray=ordered.rental_area_name.tolist())
long=by_change[['observed_change_direction','n_targets','last_mae','ridge_mae']].melt(
    id_vars=['observed_change_direction','n_targets'],var_name='method',value_name='mae')
long['method']=long.method.map({'last_mae':'Last observation','ridge_mae':'Fixed Ridge'})
long['group_label']=long.observed_change_direction+' (n='+long.n_targets.astype(str)+')'
figures['03_error_by_observed_change.html']=px.bar(
    long,x='group_label',y='mae',color='method',barmode='group',
    title='Error by observed change — retrospective groups only',
    labels={'group_label':'Actual published-median change from preceding quarter','mae':'MAE (AUD/week)','method':'Method'},
    hover_data={'n_targets':True})
figures['04_actual_versus_implied_change.html']=px.scatter(
    pairs,x='actual_change',y='ridge_implied_change',color='observed_change_direction',
    hover_name='rental_area_name',opacity=0.65,
    title='Fixed Ridge: observed change versus implied predicted change',
    labels={'actual_change':'Actual rent change (AUD/week)','ridge_implied_change':'Ridge forecast minus preceding rent (AUD/week)',
            'observed_change_direction':'Observed direction'},
    hover_data={'target_quarter':True,'last_forecast':':.2f','actual_weekly_rent_aud':':.2f',
                'ridge_forecast':':.2f','ridge_minus_last_absolute_error':':.3f','target_source_median_cell':True})
f=figures['04_actual_versus_implied_change.html']
low=float(min(pairs.actual_change.min(),pairs.ridge_implied_change.min()))-5
high=float(max(pairs.actual_change.max(),pairs.ridge_implied_change.max()))+5
f.add_trace(go.Scatter(x=[low,high],y=[low,high],mode='lines',line={'dash':'dash'},name='Perfect change prediction',hoverinfo='skip'))
f.add_hline(y=0,line_dash='dot')
f.add_annotation(x=0.99,y=0.02,xref='paper',yref='paper',text='Horizontal y=0: last-observation prediction',showarrow=False,xanchor='right')
f.update_xaxes(range=[low,high]);f.update_yaxes(range=[low,high])
for figure in figures.values():
    figure.update_layout(margin={'l':80,'r':35,'t':85,'b':95},legend={'orientation':'h','y':1.03,'x':0})
    figure.show(config={'displaylogo':False,'responsive':True})

## 8. Export only the new diagnostic evidence

Eight CSVs and four self-contained HTML charts are written under `data/interim/rental_ridge_diagnostics`. Each CSV is reloaded and checked before publication. Earlier forecasts, protocols, reports, notebooks and application files remain unchanged. No fitted models or altered forecasts are exported.

These are individually replaced files, not a database transaction. Avoid simultaneous writers and do not use outputs without the final manifest/checkpoint. Rerunning replaces this notebook's own generated report; keep personal notes elsewhere.

In [8]:
exports={
 'paired_validation_errors.csv':pairs,
 'diagnostic_metrics_overall.csv':overall_metrics,
 'error_by_lga.csv':by_lga,
 'error_by_quarter.csv':by_quarter,
 'error_by_year.csv':by_year,
 'error_by_observed_change.csv':by_change,
 'retrospective_case_examples.csv':case_examples,
 'diagnostic_summary.csv':summary,
}
for path,digest in input_hashes.items():
    require(file_hash(path)==digest,f'Protected input changed: {path.relative_to(ROOT)}')
OUT.mkdir(parents=True,exist_ok=True)
output_schemas={}
with TemporaryDirectory(prefix='.diagnostic-stage-',dir=OUT) as temporary:
    stage=Path(temporary)
    for name,frame in exports.items():
        schema={c:('int64' if pd.api.types.is_integer_dtype(frame[c]) else
                   'float64' if pd.api.types.is_numeric_dtype(frame[c]) else 'string') for c in frame.columns}
        output_schemas[name]=schema
        frame.to_csv(stage/name,index=False,encoding='utf-8',lineterminator='\n')
        loaded=pd.read_csv(stage/name,dtype=schema,keep_default_na=False)
        pd.testing.assert_frame_equal(frame.reset_index(drop=True),loaded,check_dtype=False,
                                      check_exact=False,rtol=1e-10,atol=1e-9)
    for name in exports: (stage/name).replace(OUT/name)
(OUT/'figures').mkdir(exist_ok=True)
for name,figure in figures.items():
    figure.write_html(OUT/'figures'/name,include_plotlyjs=True,full_html=True,
                      config={'responsive':True,'displaylogo':False})
(OUT/'diagnostic_protocol.json').write_text(json.dumps(plan,indent=2)+'\n',encoding='utf-8')
print('Eight CSV round trips matched; four diagnostic charts saved.')

Eight CSV round trips matched; four diagnostic charts saved.


## 9. Record what the diagnostics support — and what they do not

Observed-change groups can reveal where a fixed model's corrections help or hurt. They do not identify the cause, establish next-quarter predictability, or justify dropping hard periods. Weaknesses in a few periods are not evidence about an external event without further research.

A possible next **research hypothesis**, not a model selected or tested here, is to anchor a forecast to the last observation and model a correction/change with explicit regularisation toward small corrections. That would need a separate fixed protocol and comparison against the unchanged 589 validation targets. Simply re-expressing the target does not guarantee improvement. The existing fixed Ridge loss remains part of the experimental record.

Do not pick a different model per LGA or per realised outcome using these same validation errors and then report the selected errors as fresh performance. Keep the final period out of model selection: [scikit-learn guidance](https://scikit-learn.org/stable/common_pitfalls.html). One workbook vintage and prior exposure of final-period levels in EDA remain limitations. No production forecast is recommended.

In [9]:
created=datetime.now(timezone.utc).isoformat()
lines=['# HomeReach Melbourne — Fixed Ridge validation error diagnostics','',
       f'Generated (UTC): {created}','',
       '## Scope and evidence','',
       'Retrospective diagnosis of the saved notebook-10 predictions on 2019Q1–2023Q3: 31 LGAs, 19 quarters, 589 paired targets.',
       'The plan was specified after the overall Ridge score was known. This is development-set analysis, not a fresh independent evaluation.',
       'No models were fitted and no forecasts were replaced. All 589 targets remain in the analysis. Final period 2023Q4–2025Q3 remains unscored.',
       'The full rental history and transport tables were not loaded; existing files were hashed for integrity.',
       'Original source snapshot: '+SOURCE_HASH+'.','',
       '## Reconciled scores','',
       '| Method | Targets | MAE (AUD/week) | RMSE (AUD/week) | Signed error (forecast minus actual) |',
       '|---|---:|---:|---:|---:|']
for row in overall_metrics.itertuples(index=False):
    lines.append(f'| {row.model} | {row.n_forecasts} | {row.mae_aud_week:.4f} | {row.rmse_aud_week:.4f} | {row.mean_signed_error_aud_week:.4f} |')
lines += ['',f'Overall paired MAE difference (Ridge minus last observation): **{pooled_gap:.6f} AUD/week**.',
          f'Ridge has lower error on {wins} targets; last observation has lower error on {losses}; numerical ties: {ties}.',
          f'Ridge has lower per-LGA MAE in {int(by_lga.ridge_minus_last_mae.lt(-TOL).sum())} LGAs; last observation in {int(by_lga.ridge_minus_last_mae.gt(TOL).sum())}.','',
          '## Retrospective decomposition by observed median change','',
          '| Observed change | Targets | Last-observation MAE | Ridge MAE | Contribution to pooled MAE gap |',
          '|---|---:|---:|---:|---:|']
for row in by_change.itertuples(index=False):
    lines.append(f'| {row.observed_change_direction} | {row.n_targets} | {row.last_mae:.4f} | {row.ridge_mae:.4f} | {row.contribution_to_overall_mae_gap:.6f} |')
lines += ['', 'Each contribution divides that group’s sum of paired loss differences by all 589 targets. Contributions add to the pooled gap; negatives offset positives.',
          'Unchanged means the published median is unchanged, not every rent or property. Last observation is then exact by construction; Ridge can add error by moving away.',
          'Observed change is only known after the target. Do not turn these retrospective groups into an oracle model-selection rule.','',
          '## Time pattern','',
          '| Target year | Targets | Last-observation MAE | Ridge MAE | Ridge minus last MAE |',
          '|---|---:|---:|---:|---:|']
for row in by_year.itertuples(index=False):
    lines.append(f'| {row.target_year} | {row.n_targets} | {row.last_mae:.4f} | {row.ridge_mae:.4f} | {row.ridge_minus_last_mae:.4f} |')
lines += ['', '2023 contains Q1–Q3 only. All 19 validation quarters remain in evaluation. These are temporal associations, not explanations of their external causes.',
          'Case examples are the ten largest penalties and ten largest savings selected after seeing errors. They are not a representative sample and were not removed.','',
          '## Verification','',
          '- All 1,767 saved prediction rows checked; 1,178 previous baseline forecasts matched.',
          '- All 589 lag-1 values and source-quarter references matched to notebook-10 feature lineage.',
          '- Overall, per-LGA and per-quarter metrics recalculated and matched to saved scores.',
          '- Contributions by LGA, quarter, year and observed change reconciled to the pooled MAE gap.',
          f'- {len(self_checks)} diagnostic self-check groups passed; eight CSV round trips matched and four HTML charts saved.',
          '- No new model fits, changed forecasts, transport inputs, holdout scores or current/future forecasts.',
          '- Protected earlier research, source modules and deployed release remained unchanged.','',
          '## Interpretation and next research gate','',
          'These diagnostics describe failure patterns but do not identify a causal mechanism. Forecast errors share areas and quarters; no iid confidence interval, p-value or significance claim is made.',
          'Validation has been used repeatedly for development. Keep final-period outcomes out of model choices; they were already visible in earlier descriptive EDA, so they are not a pristine blind test.',
          'One possible controlled follow-up is a model anchored to the last observation with a learned correction. It would need its own fixed specification and can still lose to the baseline. No such candidate is fitted or selected here.',
          'Historical publication lags, revisions and boundary vintages are not reconstructed. No forecasting feature is added to the deployed explorer.','',
          '## Reproduction','',
          'Run notebooks/11_rental_ridge_error_diagnostics.ipynb on feat/rental-forecast-baselines with the existing homereach kernel.',
          'Requires the completed notebook-09 and notebook-10 outputs. Writes only data/interim/rental_ridge_diagnostics and this generated report.','',
          '## References','',
          '- https://otexts.com/fpp3/accuracy.html',
          '- https://scikit-learn.org/stable/common_pitfalls.html',
          '- https://plotly.com/python/interactive-html-export/','']
REPORT_PATH.parent.mkdir(parents=True,exist_ok=True)
REPORT_PATH.write_text('\n'.join(lines),encoding='utf-8')
for path,digest in input_hashes.items():
    require(file_hash(path)==digest,f'Protected file changed: {path.relative_to(ROOT)}')
manifest={'created_at_utc':created,'builder':'notebooks/11_rental_ridge_error_diagnostics.ipynb',
          'research_branch':BRANCH,'protocol_sha256':canonical_hash(plan),'protocol':plan,
          'ridge_manifest_sha256':input_hashes[RIDGE_MANIFEST],'baseline_manifest_sha256':input_hashes[BASE_MANIFEST],
          'input_sha256':{str(p.relative_to(ROOT)).replace('\\','/'):h for p,h in input_hashes.items()},
          'versions':{'python':sys.version.split()[0],'pandas':pd.__version__,'numpy':np.__version__,'plotly':plotly.__version__},
          'quality':{'paired_targets':len(pairs),'saved_forecast_rows':len(saved_predictions),
                     'prior_baseline_rows_verified':len(current),'lag1_matches':len(lag1),
                     'self_check_groups':len(self_checks),'new_model_fits':0,'new_forecasts':0,
                     'holdout_forecasts_scored':0,'targets_removed':0,'csv_tables':len(exports),
                     'csv_readback_matches':True,'charts':len(figures),'protected_files_unchanged':True},
          'summary':summary.to_dict(orient='records'),'self_checks':self_checks,
          'output_schemas':output_schemas,
          'output_sha256':{str(p.relative_to(OUT)).replace('\\','/'):file_hash(p)
                           for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='diagnostic_manifest.json'},
          'report_sha256':file_hash(REPORT_PATH)}
(OUT/'diagnostic_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
print('RIDGE DIAGNOSTICS CHECKPOINT PASSED')
print('Validation: 2019Q1-2023Q3 | LGAs: 31 | Paired targets:',len(pairs))
print('Saved forecast rows verified:',len(saved_predictions),'| Prior baseline rows:',len(current))
print('Lag-1 inputs independently matched:',len(lag1))
print('Previous overall/LGA/quarter metrics: matched')
print('LGA, quarter, year and observed-change contributions: reconciled')
print('Diagnostic self-check groups:',len(self_checks),'passed')
print('CSV tables saved and reloaded:',len(exports),'| Interactive charts:',len(figures))
print('New model fits: 0 | New forecasts: 0 | Final-holdout forecasts scored: 0')
print('Validation targets removed: 0 | Transport features used: False')
print('Protected inputs, earlier forecasts and deployed release unchanged: True')
print('Saved report:',REPORT_PATH.relative_to(ROOT))
print('Saved diagnostics:',OUT.relative_to(ROOT))
display(by_change.round(4))

RIDGE DIAGNOSTICS CHECKPOINT PASSED
Validation: 2019Q1-2023Q3 | LGAs: 31 | Paired targets: 589
Saved forecast rows verified: 1767 | Prior baseline rows: 1178
Lag-1 inputs independently matched: 589
Previous overall/LGA/quarter metrics: matched
LGA, quarter, year and observed-change contributions: reconciled
Diagnostic self-check groups: 12 passed
CSV tables saved and reloaded: 8 | Interactive charts: 4
New model fits: 0 | New forecasts: 0 | Final-holdout forecasts scored: 0
Validation targets removed: 0 | Transport features used: False
Protected inputs, earlier forecasts and deployed release unchanged: True
Saved report: docs\rental_ridge_diagnostics.md
Saved diagnostics: data\interim\rental_ridge_diagnostics


,observed_change_direction,n_targets,last_mae,ridge_mae,ridge_minus_last_mae,sum_paired_loss_difference,contribution_to_overall_mae_gap,ridge_lower_error_targets,last_lower_error_targets,numerical_ties
0,decreased,145,13.4345,17.7738,4.3394,629.2078,1.0683,48,97,0
1,increased,296,15.2534,14.8220,-0.4314,-127.6890,-0.2168,165,131,0
2,unchanged,148,0.0000,5.5287,5.5287,818.2402,1.3892,0,148,0


## Your checkpoint

Save the notebook with **Ctrl + S**. Send the final **RIDGE DIAGNOSTICS CHECKPOINT PASSED** output and the three-row observed-change table, or the first error.

No raw source downloads, data-pipeline runs, forecast refits or production deployment are required. Do not change notebook 09/10 settings or use the reserved final period. After this diagnosis is reviewed, any next model experiment needs a separately recorded specification.

**What you should be able to explain:** an error difference can be decomposed over identical targets; a method can help on some targets and still lose overall; actual future change is valid for retrospective diagnosis but not a known prediction input.